<a href="https://colab.research.google.com/github/Rahul9994/ML_Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
REPO_URL = "https://github.com/Rahul9994/ML_Flyrank"
REPO_DIR = "ML_Flyrank"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)

!pip install duckdb --quiet
import duckdb
from google.colab import userdata
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"

Cloning into 'ML_Flyrank'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 90 (delta 11), reused 75 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 1.84 MiB | 6.75 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [6]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{base}/fact_content_query_90d.parquet') LIMIT 5").show()

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_hash_id                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_char_count              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ query_token_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ window_start                  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ window_end                    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ impressions_90d               

In [10]:
signal_a = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1_top3'
            WHEN gsc_avg_position <= 10 THEN '2_page1'
            WHEN gsc_avg_position <= 20 THEN '3_page2'
            ELSE '4_deep'
        END AS position_bucket,
        COUNT(*) AS n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions,0)) AS avg_ctr
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY 1 ORDER BY 1
""").df()
signal_a



,position_bucket,n,avg_ctr
0,1_top3,727362,0.004756
1,2_page1,1456122,0.003473
2,3_page2,519223,0.002770
3,4_deep,908354,0.001289


In [11]:
signal_b = con.sql(f"""
    SELECT
        CASE
            WHEN content_total_impressions_90d >= 1000 THEN '1_high_volume'
            WHEN content_total_impressions_90d >= 100 THEN '2_mid_volume'
            ELSE '3_low_volume'
        END AS volume_bucket,
        COUNT(*) AS n,
        AVG(clicks_90d * 1.0 / NULLIF(impressions_90d,0)) AS avg_ctr
    FROM read_parquet('{base}/fact_content_query_90d.parquet')
    WHERE impressions_90d > 0
    GROUP BY 1 ORDER BY 1
""").df()
signal_b

,volume_bucket,n,avg_ctr
0,1_high_volume,2215563,0.002144
1,2_mid_volume,189470,0.000784
2,3_low_volume,9215,0.000466


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

pages ranking well (low avg_position_90d) with real search volume (content_total_impressions_90d) but low observed CTR are flagged highest — they're visible and searched-for, but under-converting on clicks, which is exactly a metadata/snippet review case, not a ranking problem.

In [12]:
queue = con.sql(f"""
    WITH expected AS (
        SELECT
            CASE
                WHEN avg_position_90d <= 3 THEN 727362
                WHEN avg_position_90d <= 10 THEN 1456122
                WHEN avg_position_90d <= 20 THEN 519223
                ELSE 908354
            END AS bucket_n,
            *
        FROM read_parquet('{base}/fact_content_query_90d.parquet')
        WHERE impressions_90d > 0
    )
    SELECT
        client_hash_id, content_hash_id, query_hash_id,
        avg_position_90d, content_total_impressions_90d,
        clicks_90d * 1.0 / NULLIF(impressions_90d,0) AS ctr,
        -- score: rewards good position + real volume, penalizes underperforming CTR
        (1.0 / NULLIF(avg_position_90d,0)) * LOG(1 + content_total_impressions_90d)
            - (clicks_90d * 1.0 / NULLIF(impressions_90d,0)) AS score,
        'ctr_below_position_and_volume_expectation' AS reason_code,
        'review_snippet_metadata' AS action_label
    FROM expected
    ORDER BY score DESC
""").df()

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows")
queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 2414248 rows


,client_hash_id,content_hash_id,query_hash_id,avg_position_90d,content_total_impressions_90d,ctr,score,reason_code,action_label
0,client_23a62021009f63c4,content_b79e7694245faa61,query_bb441b804cbdff7e,0.000491,12050,0.000000,8313.044027,ctr_below_position_and_volume_expectation,review_snippet_metadata
1,client_73cda7b4e4f265ea,content_d252f5b1b1b4d5ff,query_05118e04631b9763,0.000683,43980,0.000000,6797.740106,ctr_below_position_and_volume_expectation,review_snippet_metadata
2,client_23a62021009f63c4,content_a76ba8dff727e4b1,query_89b40475b91adf83,0.000615,7795,0.000000,6330.129532,ctr_below_position_and_volume_expectation,review_snippet_metadata
3,client_20259bd6705d81d4,content_f9f6162126731e51,query_8fc256370f0f9d6e,0.000717,12376,0.000000,5709.198470,ctr_below_position_and_volume_expectation,review_snippet_metadata
4,client_23a62021009f63c4,content_903b044491360e4b,query_a8602fc707cb5b0b,0.000730,9835,0.000000,5470.161372,ctr_below_position_and_volume_expectation,review_snippet_metadata
5,client_23a62021009f63c4,content_ee20db622c90e580,query_7534c6b3bb504de0,0.000760,7632,0.000000,5106.714944,ctr_below_position_and_volume_expectation,review_snippet_metadata
6,client_73cda7b4e4f265ea,content_2909a3abb1364aba,query_7562d8fc51b223ca,0.000687,2420,0.002747,4927.093666,ctr_below_position_and_volume_expectation,review_snippet_metadata
7,client_23a62021009f63c4,content_300690ed36a233d4,query_e046cc0cd43d26ac,0.001092,13135,0.000000,3772.512236,ctr_below_position_and_volume_expectation,review_snippet_metadata
8,client_73cda7b4e4f265ea,content_474c3e4d7baeb906,query_7628d9b9606d06e1,0.001363,47002,0.000000,3427.004111,ctr_below_position_and_volume_expectation,review_snippet_metadata
9,client_73cda7b4e4f265ea,content_0d23b141ae61ef08,query_6914bca53e34890c,0.000996,2173,0.000000,3350.608578,ctr_below_position_and_volume_expectation,review_snippet_metadata


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")

Wrote 2414248 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
queue_by_content = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        AVG(avg_position_90d) AS avg_position_90d,
        MAX(content_total_impressions_90d) AS content_total_impressions_90d,
        SUM(clicks_90d) * 1.0 / NULLIF(SUM(impressions_90d),0) AS ctr,
        (1.0 / GREATEST(AVG(avg_position_90d), 1.0)) * LOG(1 + MAX(content_total_impressions_90d))
            - (SUM(clicks_90d) * 1.0 / NULLIF(SUM(impressions_90d),0)) AS score
    FROM read_parquet('{base}/fact_content_query_90d.parquet')
    WHERE impressions_90d > 0 AND avg_position_90d >= 1
    GROUP BY client_hash_id, content_hash_id
    ORDER BY score DESC
""").df()

queue_by_content.head(10)

,client_hash_id,content_hash_id,avg_position_90d,content_total_impressions_90d,ctr,score
0,client_a80fca3f171ed1de,content_b040ac222041ef39,1.013647,16230,0.000000,4.153659
1,client_20259bd6705d81d4,content_0764a42f5dcf0c1b,1.000000,8216,0.000000,3.914713
2,client_62f4a7e64f5e0096,content_5cdc3f41e57cba3e,1.000000,5404,0.000000,3.732796
3,client_73cda7b4e4f265ea,content_bf1200633240b876,1.000000,5302,0.008333,3.716188
4,client_73cda7b4e4f265ea,content_8856de93355df4d8,1.010000,4427,0.000000,3.610107
5,client_a80fca3f171ed1de,content_682c6a15661438ff,1.000000,3595,0.000000,3.555820
6,client_73cda7b4e4f265ea,content_4c40537ed0da4b6f,1.221239,21685,0.007937,3.542704
7,client_1a8bf67cad4ee525,content_044ab2dcf9397076,1.000000,3201,0.000000,3.505421
8,client_a80fca3f171ed1de,content_a1e639626a441179,1.107133,7334,0.000000,3.491361
9,client_62f4a7e64f5e0096,content_70487b449ce643aa,1.000000,3083,0.000000,3.489114


In [16]:
queue_clean = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id, query_hash_id,
        avg_position_90d, content_total_impressions_90d,
        clicks_90d * 1.0 / NULLIF(impressions_90d,0) AS ctr,
        (1.0 / GREATEST(avg_position_90d, 1.0)) * LOG(1 + content_total_impressions_90d)
            - (clicks_90d * 1.0 / NULLIF(impressions_90d,0)) AS score,
        'ctr_below_position_and_volume_expectation' AS reason_code,
        'review_snippet_metadata' AS action_label
    FROM read_parquet('{base}/fact_content_query_90d.parquet')
    WHERE impressions_90d > 0 AND avg_position_90d >= 1
    ORDER BY score DESC
""").df()

queue_clean.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,query_hash_id,avg_position_90d,content_total_impressions_90d,ctr,score,reason_code,action_label
0,client_e547b89c05043229,content_eadb33b5df496f4a,query_001172a6d2773bf3,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata
1,client_e547b89c05043229,content_eadb33b5df496f4a,query_001207cbcf2c1091,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata
2,client_e547b89c05043229,content_eadb33b5df496f4a,query_00b88b08c9212a18,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata
3,client_e547b89c05043229,content_eadb33b5df496f4a,query_018b5f588d1039ca,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata
4,client_e547b89c05043229,content_eadb33b5df496f4a,query_01a3b87697dedb66,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata
5,client_e547b89c05043229,content_eadb33b5df496f4a,query_01f123030eb0a8ed,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata
6,client_e547b89c05043229,content_eadb33b5df496f4a,query_02c9ee374db9ae78,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata
7,client_e547b89c05043229,content_eadb33b5df496f4a,query_02e870ce5bb92e6c,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata
8,client_e547b89c05043229,content_eadb33b5df496f4a,query_03220c1bc4a3ca86,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata
9,client_e547b89c05043229,content_eadb33b5df496f4a,query_03289efa0db967b9,1.0,1961517,0.0,6.292592,ctr_below_position_and_volume_expectation,review_snippet_metadata


In [14]:
top20 = queue.head(20)
for i, row in top20.iterrows():
    confidence = "high" if row['content_total_impressions_90d'] > 500 else "low-volume, less certain"
    print(f"Row {i}: reason={row['reason_code']}, action={row['action_label']}, "
          f"position={row['avg_position_90d']:.1f}, volume={row['content_total_impressions_90d']}, "
          f"ctr={row['ctr']:.4f}, confidence={confidence}")
    print("  -> What would make this wrong: if rare_impressions_share or "
          "anonymized_impressions_share is high for this row, the CTR/volume numbers "
          "are noisier than they look, and this pick could be a measurement artifact.\n")

Row 0: reason=ctr_below_position_and_volume_expectation, action=review_snippet_metadata, position=0.0, volume=12050, ctr=0.0000, confidence=high
  -> What would make this wrong: if rare_impressions_share or anonymized_impressions_share is high for this row, the CTR/volume numbers are noisier than they look, and this pick could be a measurement artifact.

Row 1: reason=ctr_below_position_and_volume_expectation, action=review_snippet_metadata, position=0.0, volume=43980, ctr=0.0000, confidence=high
  -> What would make this wrong: if rare_impressions_share or anonymized_impressions_share is high for this row, the CTR/volume numbers are noisier than they look, and this pick could be a measurement artifact.

Row 2: reason=ctr_below_position_and_volume_expectation, action=review_snippet_metadata, position=0.0, volume=7795, ctr=0.0000, confidence=high
  -> What would make this wrong: if rare_impressions_share or anonymized_impressions_share is high for this row, the CTR/volume numbers are no

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Leakage check: no future-window or label-derived inputs were used. Features are
avg_position_90d and content_total_impressions_90d — both observed independently
of the ctr label at the same point in time, no forward-looking data used.

My score uses content_total_impressions_90d, which is a content-level aggregate
repeated across every query row for that content. This causes one high-volume
content item to dominate the ranked queue with many near-identical rows (same
position, same score) rather than surfacing genuinely distinct opportunities.
A cleaner version would rank at the content level first, then show top queries
within each content item — or use a query-level volume field instead.

In [15]:
weak = queue.head(20)[["content_hash_id", "content_total_impressions_90d"]]
print(weak.sort_values("content_total_impressions_90d").head(3))

             content_hash_id  content_total_impressions_90d
9   content_0d23b141ae61ef08                           2173
6   content_2909a3abb1364aba                           2420
10  content_3279c6c62a2f0084                           2815


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.